# SIEWS+ 5.0 — Multi-Stage YOLO Training (Full Kaggle Pipeline)
**AI Safety Monitoring for Oil & Gas (Migas)**

This notebook runs the **complete pipeline on Kaggle** — no local processing needed.

**Stages trained:**
- Stage 1: Person Detection (YOLO26n — pretrained COCO, no training)
- Stage 2: PPE + Safety Harness (YOLO26s — fine-tune, 9 classes)
- Stage 3: Fire + Smoke (YOLO26n — fine-tune, 2 classes)
- Stage 4: Infrastructure + Vehicles + Equipment (YOLO26s — fine-tune, 8 classes)

## Setup: Upload Raw Datasets to Kaggle Once

Upload **all 16 raw ZIP files** from `dataset/` as a Kaggle dataset named **`migas-siews-raw`**:
```
Airbus Oil Storage Detection.zip
CCTV.v2i.yolo26.zip
Construction Equipment.v2i.yolo26.zip
Construction Safety - Open hole - Excavation Detection.v5-detr.yolo26.zip
Fire and Smoke.v14i.yolo26.zip
Oil and Gas Safety.v3i.yolo26.zip
Open hole Detection - Construction Safety.v14-final-training.yolo26.zip
Safety Harness Dataset.v1i.yolo26.zip
Yolo Truck.v2i.yolo26.zip
analog_pressure_gauge.v21i.yolo26.zip
harness.v4i.yolo26.zip
no safety harness.v2i.yolo26.zip
number and adr plate detector.v3i.yolo26.zip
oil storage tanker detection.v1i.yolo26.zip
oil tank truck.v1i.yolo26.zip
oil-and-gas.v1-dinnna.yolo26.zip
```

Then attach **`migas-siews-raw`** to this notebook via *Add data* → search `migas-siews-raw`.

**GPU recommended:** T4 x2 or P100

## 0. Install Dependencies

In [ ]:
!pip install ultralytics kaggle roboflow -q
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Setup & Extract Raw Datasets

In [ ]:
import os, zipfile, shutil, random, hashlib, yaml as _yaml
from pathlib import Path
from collections import defaultdict, Counter

random.seed(42)

WORK_DIR    = Path('/kaggle/working')
EXTRACT_DIR = WORK_DIR / 'extracted'
STAGE2_DIR  = WORK_DIR / 'stage2'
STAGE3_DIR  = WORK_DIR / 'stage3'
STAGE4_DIR  = WORK_DIR / 'stage4'

for d in [EXTRACT_DIR, STAGE2_DIR, STAGE3_DIR, STAGE4_DIR]:
    d.mkdir(exist_ok=True)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Keywords that appear in our 16 dataset names
KNOWN_KW = [
    'airbus oil storage', 'cctv', 'construction equipment', 'construction safety',
    'fire and smoke', 'oil and gas safety', 'open hole detection', 'safety harness',
    'yolo truck', 'analog_pressure', 'harness', 'no safety harness',
    'number and adr', 'oil storage tanker', 'oil tank truck', 'oil-and-gas',
]

# ── Detect RAW_INPUT: walk ALL levels of /kaggle/input ────────────────────────
# Kaggle stores user datasets at nested paths like:
#   /kaggle/input/datasets/<user>/<dataset-slug>/   (extracted folders)
#   /kaggle/input/<dataset-slug>/                   (ZIP files)
RAW_INPUT = None

for root, dirs, files in os.walk('/kaggle/input'):
    dirs_lower  = [d.lower() for d in dirs]
    files_lower = [f.lower() for f in files]

    # Match extracted folder format: subdirs contain known dataset names
    dir_hits = sum(1 for kw in KNOWN_KW if any(kw in d for d in dirs_lower))
    if dir_hits >= 3:
        RAW_INPUT = Path(root)
        dirs.clear()   # stop recursing
        break

    # Match ZIP format: files contain known dataset names
    zip_hits = sum(1 for kw in KNOWN_KW if any(kw in f for f in files_lower if f.endswith('.zip')))
    if zip_hits >= 3:
        RAW_INPUT = Path(root)
        dirs.clear()
        break

# ── Report ────────────────────────────────────────────────────────────────────
if RAW_INPUT:
    has_zips = any(f.endswith('.zip') for f in os.listdir(RAW_INPUT))
    fmt = 'ZIP files' if has_zips else 'extracted folders'
    items = sorted(RAW_INPUT.iterdir())
    print(f'RAW_INPUT = {RAW_INPUT}  [{fmt}]')
    print(f'Found {len(items)} items:')
    for item in items[:20]:
        tag = 'dir' if item.is_dir() else f'{item.stat().st_size//1024//1024}MB'
        print(f'  {item.name[:70]}  [{tag}]')
else:
    print('ERROR: dataset not found. Tree dump:')
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth > 5:
            dirs.clear(); continue
        print('  ' * depth + Path(root).name + f'/  ({len(dirs)} dirs, {len(files)} files)')
    raise RuntimeError('Dataset not found — attach migas_siews_v1 via Add data')

In [ ]:
# ── Setup EXTRACT_DIR based on RAW_INPUT format ───────────────────────────────
has_zips = any(f.endswith('.zip') for f in os.listdir(RAW_INPUT))
has_dirs = any(d.is_dir() for d in RAW_INPUT.iterdir())

def is_extracted(path: Path) -> bool:
    return path.exists() and any(
        True for f in path.rglob('*.*') if f.suffix.lower() in IMG_EXTS
    )

if not has_zips and has_dirs:
    # ── Extracted folders already — point EXTRACT_DIR straight at source ─────
    EXTRACT_DIR = RAW_INPUT
    print(f'Using pre-extracted folders from: {EXTRACT_DIR}')

else:
    # ── ZIP format — extract each one ─────────────────────────────────────────
    zips = sorted(RAW_INPUT.glob('*.zip'))
    print(f'Extracting {len(zips)} ZIPs to {EXTRACT_DIR} ...\n')
    for zip_path in zips:
        dst = EXTRACT_DIR / zip_path.stem
        if is_extracted(dst):
            n = len([f for f in dst.rglob('*.*') if f.suffix.lower() in IMG_EXTS])
            print(f'  [SKIP] {zip_path.stem[:60]}  ({n} imgs)')
            continue
        print(f'  Extracting {zip_path.name} ...')
        dst.mkdir(exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(dst)
        n = len([f for f in dst.rglob('*.*') if f.suffix.lower() in IMG_EXTS])
        print(f'  → {n} images')

# ── Summary ───────────────────────────────────────────────────────────────────
print('\n─── Dataset Summary ───')
total, ds_dirs = 0, [d for d in sorted(EXTRACT_DIR.iterdir()) if d.is_dir()]
for d in ds_dirs:
    n = len([f for f in d.rglob('*.*') if f.suffix.lower() in IMG_EXTS])
    total += n
    print(f'  {d.name[:65]:65s}  {n:>5} imgs')
print(f'\nTotal: {total} images across {len(ds_dirs)} datasets')
assert len(ds_dirs) >= 3, f'Only {len(ds_dirs)} dataset dirs found — check RAW_INPUT!'
print('OK ✓')

In [ ]:
# ── Merge Helper Functions ────────────────────────────────────────────────────

def file_hash(path: Path) -> str:
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def ensure_split_dirs(stage_dir: Path):
    for split in ['train', 'val', 'test']:
        (stage_dir / split / 'images').mkdir(parents=True, exist_ok=True)
        (stage_dir / split / 'labels').mkdir(parents=True, exist_ok=True)

def find_split_dirs(root: Path) -> dict:
    """Return {target_split: (img_dir, lbl_dir)}.  Normalises 'valid' → 'val'."""
    splits = {}
    for src_split, dst_split in [('train','train'), ('valid','val'), ('val','val'), ('test','test')]:
        if dst_split in splits:
            continue
        img_dir = root / src_split / 'images'
        lbl_dir = root / src_split / 'labels'
        if img_dir.exists() and lbl_dir.exists():
            splits[dst_split] = (img_dir, lbl_dir)
    # Flat layout fallback (images/ labels/ at root)
    if not splits:
        img_dir = root / 'images'
        lbl_dir = root / 'labels'
        if img_dir.exists() and lbl_dir.exists():
            splits['_flat'] = (img_dir, lbl_dir)
    return splits

def collect_samples(img_dir: Path, lbl_dir: Path) -> list:
    samples = []
    for img in img_dir.glob('*.*'):
        if img.suffix.lower() not in IMG_EXTS:
            continue
        lbl = lbl_dir / (img.stem + '.txt')
        if lbl.exists():
            samples.append({'image': img, 'label': lbl})
    return samples

def remap_and_copy_label(label_file: Path, class_map: dict, dst: Path) -> bool:
    """Remap class IDs; return True if at least one valid annotation written."""
    lines  = label_file.read_text().strip().splitlines()
    output = []
    for line in lines:
        parts = line.strip().split()
        if not parts or not parts[0].lstrip('-').isdigit():
            continue
        cls = int(parts[0])
        if cls in class_map:
            output.append(f'{class_map[cls]} ' + ' '.join(parts[1:]))
    if output:
        dst.write_text('\n'.join(output))
        return True
    return False

print('Merge helper functions defined ✓')

## 2. Merge & Remap Datasets

In [ ]:
# ── Target Classes ────────────────────────────────────────────────────────────
STAGE2_CLASSES = {0:'helmet', 1:'no_helmet', 2:'safety_vest', 3:'no_vest',
                  4:'safety_harness', 5:'no_harness', 6:'gloves', 7:'boots', 8:'goggles'}
STAGE3_CLASSES = {0:'fire', 1:'smoke'}
STAGE4_CLASSES = {0:'oil_storage_tank', 1:'oil_tank_truck', 2:'construction_equip',
                  3:'open_hole', 4:'pressure_gauge', 5:'adr_plate', 6:'truck', 7:'cctv_anomaly'}

# ── Manual class mapping overrides keyed by dataset name substring ─────────────
# {name_substring: {'stage': 'stage2'|'stage3'|'stage4', 'map': {src_id: dst_id}}}
MANUAL_CLASS_OVERRIDES = {
    # ── Stage 2: PPE + Safety Harness ────────────────────────────────────────
    'safety harness dataset': {'stage':'stage2', 'map':{0:4}},
    'harness.v4i':            {'stage':'stage2', 'map':{0:4}},
    'no safety harness':      {'stage':'stage2', 'map':{0:4, 1:5}},
    'oil and gas safety': {
        'stage': 'stage2',
        'map':   {0:0, 1:2, 3:1, 4:3, 5:4, 6:5, 7:7},  # skip class 2 (person)
    },
    # ── Stage 3: Fire + Smoke ─────────────────────────────────────────────────
    'fire and smoke':      {'stage':'stage3', 'map':{0:0, 1:1}},
    'oil-and-gas.v1-dinnna': {'stage':'stage3', 'map':{0:0, 1:1}},
    # ── Stage 4: Infrastructure + Vehicles + Equipment ────────────────────────
    'airbus oil storage detection':      {'stage':'stage4', 'map':{0:0}},
    'oil storage tanker detection':      {'stage':'stage4', 'map':{0:0}},
    'oil tank truck':                    {'stage':'stage4', 'map':{0:1}},
    'construction equipment': {
        'stage':'stage4',
        'map': {i:2 for i in range(14)},   # all 14 classes → construction_equip
    },
    'construction safety - open hole':   {'stage':'stage4', 'map':{3:3}},
    'open hole detection':               {'stage':'stage4', 'map':{4:3}},
    'analog_pressure_gauge':             {'stage':'stage4', 'map':{0:4, 1:4, 2:4, 3:4}},
    'yolo truck':                        {'stage':'stage4', 'map':{0:6, 1:6}},
    'number and adr plate':              {'stage':'stage4', 'map':{i:5 for i in range(5)}},
    'cctv':                              {'stage':'stage4', 'map':{0:7, 1:7}},
}

# ── Keyword fallback for auto-detection from data.yaml ────────────────────────
S2_KEYWORD_MAP = {
    'helmet':0, 'hard-hat':0, 'hardhat':0, 'hard_hat':0,
    'no-helmet':1, 'no_helmet':1, 'nohelmet':1, 'no-hardhat':1,
    'vest':2, 'safety-vest':2, 'safety_vest':2, 'hi-vis':2,
    'no-vest':3, 'no_vest':3, 'novest':3,
    'harness':4, 'safety-harness':4, 'safety_harness':4,
    'no-harness':5, 'no_harness':5, 'noharness':5,
    'gloves':6, 'boots':7, 'goggles':8,
}
S3_KEYWORD_MAP = {'fire':0, 'flame':0, 'smoke':1, 'fumes':1}

print('Class mappings defined:')
print(f'  Stage 2: {len(STAGE2_CLASSES)} classes')
print(f'  Stage 3: {len(STAGE3_CLASSES)} classes')
print(f'  Stage 4: {len(STAGE4_CLASSES)} classes')
print(f'  Overrides: {len(MANUAL_CLASS_OVERRIDES)} entries')

In [ ]:
# ── Stage Detection + Merge Function ─────────────────────────────────────────

def detect_stage_and_map(ds_path: Path):
    """
    Return (stage, class_map) for a dataset directory.
    Tries MANUAL_CLASS_OVERRIDES first, then keyword scan of data.yaml.
    Returns (None, None) if unknown.
    """
    name_lower = ds_path.name.lower()

    # 1. Manual overrides (highest priority)
    for key, override in MANUAL_CLASS_OVERRIDES.items():
        if key.lower() in name_lower:
            return override['stage'], override['map']

    # 2. Read data.yaml and try keyword detection
    yaml_path = next(
        (ds_path / f for f in ['data.yaml', 'dataset.yaml'] if (ds_path / f).exists()),
        None
    )
    if yaml_path is None:
        return None, None

    with open(yaml_path) as f:
        data = _yaml.safe_load(f) or {}

    names = data.get('names', [])
    if isinstance(names, dict):
        names = list(names.values())

    s2_map, s3_map = {}, {}
    for idx, cls_name in enumerate(names):
        kw = str(cls_name).lower().replace('-', '_').replace(' ', '_')
        if kw in S2_KEYWORD_MAP:
            s2_map[idx] = S2_KEYWORD_MAP[kw]
        elif kw in S3_KEYWORD_MAP:
            s3_map[idx] = S3_KEYWORD_MAP[kw]

    if s3_map:
        return 'stage3', s3_map
    if s2_map:
        return 'stage2', s2_map
    return None, None


def merge_stage(datasets: list, stage_dir: Path, stage_name: str) -> dict:
    """
    datasets: list of (ds_path: Path, class_map: dict)
    Merges all datasets into stage_dir with short filenames.
    Returns count dict {'train': N, 'val': N, 'test': N}.
    """
    if stage_dir.exists():
        shutil.rmtree(stage_dir)
    ensure_split_dirs(stage_dir)

    seen_hashes = set()
    counts      = defaultdict(int)

    for ds_path, class_map in datasets:
        split_dirs = find_split_dirs(ds_path)
        if not split_dirs:
            print(f'  [WARN] {ds_path.name}: no split dirs found, skipping.')
            continue

        print(f'\n  Processing: {ds_path.name}')
        print(f'    Mapping:   {class_map}')

        for split_key, (img_dir, lbl_dir) in split_dirs.items():
            samples = collect_samples(img_dir, lbl_dir)
            if not samples:
                continue

            # Flat layout → random 80/10/10 split
            if split_key == '_flat':
                random.shuffle(samples)
                n  = len(samples)
                t  = int(n * 0.8)
                v  = int(n * 0.1)
                manual_splits = {
                    'train': samples[:t],
                    'val':   samples[t:t+v],
                    'test':  samples[t+v:],
                }
            else:
                manual_splits = {split_key: samples}

            for target_split, split_samples in manual_splits.items():
                added = 0
                for s in split_samples:
                    h = file_hash(s['image'])
                    if h in seen_hashes:
                        continue
                    seen_hashes.add(h)

                    # Short name: 20-char slug + 12-char content hash → max ~35 chars
                    slug = ds_path.name[:20].replace(' ', '_').replace('/', '_')
                    stem = f'{slug}_{h[:12]}'

                    dst_img = stage_dir / target_split / 'images' / (stem + s['image'].suffix)
                    shutil.copy2(s['image'], dst_img)

                    dst_lbl = stage_dir / target_split / 'labels' / (stem + '.txt')
                    if remap_and_copy_label(s['label'], class_map, dst_lbl):
                        added += 1
                    else:
                        dst_img.unlink(missing_ok=True)

                counts[target_split] += added
                if added:
                    print(f'    {target_split}: +{added}')

    print(f'\n  TOTAL {stage_name}: train={counts["train"]}, val={counts["val"]}, test={counts["test"]}')

    # Data leak check
    train_names = set(p.name for p in (stage_dir / 'train' / 'images').glob('*.*'))
    val_names   = set(p.name for p in (stage_dir / 'val'   / 'images').glob('*.*'))
    overlap = train_names & val_names
    if overlap:
        print(f'  WARNING: {len(overlap)} name collision(s) between train and val!')
    else:
        print(f'  Data leak check: OK (0 overlap) ✓')

    return dict(counts)

print('detect_stage_and_map() and merge_stage() defined ✓')

In [ ]:
# ── Detect stage for each extracted dataset ───────────────────────────────────
stage2_datasets, stage3_datasets, stage4_datasets, unknown = [], [], [], []

for ds_path in sorted(EXTRACT_DIR.iterdir()):
    if not ds_path.is_dir():
        continue
    stage, class_map = detect_stage_and_map(ds_path)
    if   stage == 'stage2': stage2_datasets.append((ds_path, class_map))
    elif stage == 'stage3': stage3_datasets.append((ds_path, class_map))
    elif stage == 'stage4': stage4_datasets.append((ds_path, class_map))
    else:                   unknown.append(ds_path.name)

print(f'Stage 2: {len(stage2_datasets)} datasets')
for ds, m in stage2_datasets:
    print(f'  {ds.name}  →  {m}')

print(f'\nStage 3: {len(stage3_datasets)} datasets')
for ds, m in stage3_datasets:
    print(f'  {ds.name}  →  {m}')

print(f'\nStage 4: {len(stage4_datasets)} datasets')
for ds, m in stage4_datasets:
    print(f'  {ds.name}  →  {m}')

if unknown:
    print(f'\nUnknown / skipped ({len(unknown)}):')
    for u in unknown:
        print(f'  {u}')

assert len(stage2_datasets) > 0, 'No Stage 2 datasets found!'
assert len(stage3_datasets) > 0, 'No Stage 3 datasets found!'
assert len(stage4_datasets) > 0, 'No Stage 4 datasets found!'
print('\nDetection OK ✓')

print('='*70); print('Merging Stage 2 — PPE (9 cls)'); print('='*70)
s2_counts = merge_stage(stage2_datasets, STAGE2_DIR, 'stage2')

print('='*70); print('Merging Stage 3 — Fire+Smoke (2 cls)'); print('='*70)
s3_counts = merge_stage(stage3_datasets, STAGE3_DIR, 'stage3')

print('='*70); print('Merging Stage 4 — Infra (8 cls)'); print('='*70)
s4_counts = merge_stage(stage4_datasets, STAGE4_DIR, 'stage4')

print(f'\nStage 2: {s2_counts}')
print(f'Stage 3: {s3_counts}')
print(f'Stage 4: {s4_counts}')

In [ ]:
## 3. Write YAML Configs

In [ ]:
# ── Write YAML Configs ────────────────────────────────────────────────────────
stage2_yaml = WORK_DIR / 'stage2.yaml'
stage2_yaml.write_text(f"""path: {STAGE2_DIR}
train: train/images
val:   val/images
test:  test/images
nc: 9
names:
  0: helmet
  1: no_helmet
  2: safety_vest
  3: no_vest
  4: safety_harness
  5: no_harness
  6: gloves
  7: boots
  8: goggles
""")

stage3_yaml = WORK_DIR / 'stage3.yaml'
stage3_yaml.write_text(f"""path: {STAGE3_DIR}
train: train/images
val:   val/images
test:  test/images
nc: 2
names:
  0: fire
  1: smoke
""")

stage4_yaml = WORK_DIR / 'stage4.yaml'
stage4_yaml.write_text(f"""path: {STAGE4_DIR}
train: train/images
val:   val/images
test:  test/images
nc: 8
names:
  0: oil_storage_tank
  1: oil_tank_truck
  2: construction_equip
  3: open_hole
  4: pressure_gauge
  5: adr_plate
  6: truck
  7: cctv_anomaly
""")

print('YAML configs written:')
print(f'  {stage2_yaml}  (train={s2_counts.get("train",0)}, val={s2_counts.get("val",0)})')
print(f'  {stage3_yaml}  (train={s3_counts.get("train",0)}, val={s3_counts.get("val",0)})')
print(f'  {stage4_yaml}  (train={s4_counts.get("train",0)}, val={s4_counts.get("val",0)})')

In [ ]:
## 4. Verify Merged Datasets

In [ ]:
# ── Verify All Merged Stages ──────────────────────────────────────────────────
def count_images(d: Path) -> int:
    if not d.exists():
        return 0
    return len([f for f in d.glob('*.*') if f.suffix.lower() in IMG_EXTS])

def verify_stage(name: str, sdir: Path):
    n_train = count_images(sdir / 'train' / 'images')
    n_val   = count_images(sdir / 'val'   / 'images')
    n_test  = count_images(sdir / 'test'  / 'images')

    if n_train == 0 or n_val == 0:
        print(f'\n{name} FAILED — contents:')
        for sp in ['train', 'val', 'test']:
            sp_dir = sdir / sp / 'images'
            files  = list(sp_dir.glob('*.*')) if sp_dir.exists() else []
            print(f'  {sp}/images: exists={sp_dir.exists()}, count={len(files)}')

    assert n_train > 0, f'{name}: no training images!'
    assert n_val   > 0, f'{name}: no validation images!'

    train_names = set(p.name for p in (sdir / 'train' / 'images').glob('*.*'))
    val_names   = set(p.name for p in (sdir / 'val'   / 'images').glob('*.*'))
    overlap = train_names & val_names
    assert len(overlap) == 0, f'{name} DATA LEAK: {len(overlap)} files in both train+val!'

    print(f'{name}: train={n_train}, val={n_val}, test={n_test} | Leak check: OK ✓')

verify_stage('Stage 2 (PPE)',           STAGE2_DIR)
verify_stage('Stage 3 (Fire+Smoke)',    STAGE3_DIR)
verify_stage('Stage 4 (Infra)',         STAGE4_DIR)
print('\nAll datasets verified ✓  — ready to train.')

## 3a. Train Stage 2 — PPE + Safety Harness (YOLO26s, 9 classes)

In [ ]:
from ultralytics import YOLO

s2_train_n = len(list((STAGE2_DIR / 'train' / 'images').glob('*.*')))
s2_val_n   = len(list((STAGE2_DIR / 'val'   / 'images').glob('*.*')))
print(f'Stage 2: train={s2_train_n}, val={s2_val_n} images')
assert s2_train_n > 0 and s2_val_n > 0, 'Stage 2 dataset not ready!'

train_names_s2 = set(p.name for p in (STAGE2_DIR / 'train' / 'images').glob('*.*'))
val_names_s2   = set(p.name for p in (STAGE2_DIR / 'val'   / 'images').glob('*.*'))
assert len(train_names_s2 & val_names_s2) == 0, 'Stage 2 data leak detected!'
print('Data leak check: OK ✓')

model_s2 = YOLO('yolo26s.pt')
results_s2 = model_s2.train(
    data=str(stage2_yaml),
    epochs=150,
    batch=16,
    imgsz=640,
    patience=40,
    project='/kaggle/working/runs',
    name='stage2_ppe_harness',
    pretrained=True,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    degrees=15.0,
    scale=0.3,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    erasing=0.3,
    dropout=0.1,
    weight_decay=0.001,
    label_smoothing=0.1,
    optimizer='SGD',
    lr0=0.01,
    lrf=0.001,
    warmup_epochs=5,
    cos_lr=True,
    momentum=0.937,
    multi_scale=False,
    verbose=True,
    save_period=10,
    plots=True,
)

metrics_s2 = results_s2.results_dict
print(f"Stage 2 mAP50:    {metrics_s2.get('metrics/mAP50(B)', 'N/A')}")
print(f"Stage 2 mAP50-95: {metrics_s2.get('metrics/mAP50-95(B)', 'N/A')}")

## 3b. Train Stage 3 — Fire + Smoke (YOLO26n, 2 classes)

In [ ]:
s3_train_n = len(list((STAGE3_DIR / 'train' / 'images').glob('*.*')))
s3_val_n   = len(list((STAGE3_DIR / 'val'   / 'images').glob('*.*')))
print(f'Stage 3: train={s3_train_n}, val={s3_val_n} images')
assert s3_train_n > 0 and s3_val_n > 0, 'Stage 3 dataset not ready!'

train_names_s3 = set(p.name for p in (STAGE3_DIR / 'train' / 'images').glob('*.*'))
val_names_s3   = set(p.name for p in (STAGE3_DIR / 'val'   / 'images').glob('*.*'))
assert len(train_names_s3 & val_names_s3) == 0, 'Stage 3 data leak detected!'
print('Data leak check: OK ✓')

model_s3 = YOLO('yolo26n.pt')
results_s3 = model_s3.train(
    data=str(stage3_yaml),
    epochs=150,
    batch=32,
    imgsz=640,
    patience=40,
    project='/kaggle/working/runs',
    name='stage3_environment',
    pretrained=True,
    mosaic=1.0,
    mixup=0.2,
    degrees=10.0,
    scale=0.3,
    fliplr=0.5,
    flipud=0.1,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
    erasing=0.4,
    dropout=0.1,
    weight_decay=0.001,
    label_smoothing=0.1,
    optimizer='SGD',
    lr0=0.01,
    lrf=0.001,
    warmup_epochs=5,
    cos_lr=True,
    momentum=0.937,
    multi_scale=False,
    verbose=True,
    save_period=10,
    plots=True,
)

metrics_s3 = results_s3.results_dict
print(f"Stage 3 mAP50:    {metrics_s3.get('metrics/mAP50(B)', 'N/A')}")
print(f"Stage 3 mAP50-95: {metrics_s3.get('metrics/mAP50-95(B)', 'N/A')}")

## 3c. Train Stage 4 — Infrastructure + Vehicles + Equipment (YOLO26s, 8 classes)

In [ ]:
s4_train_n = len(list((STAGE4_DIR / 'train' / 'images').glob('*.*')))
s4_val_n   = len(list((STAGE4_DIR / 'val'   / 'images').glob('*.*')))
print(f'Stage 4: train={s4_train_n}, val={s4_val_n} images')
assert s4_train_n > 0 and s4_val_n > 0, 'Stage 4 dataset not ready!'

train_names_s4 = set(p.name for p in (STAGE4_DIR / 'train' / 'images').glob('*.*'))
val_names_s4   = set(p.name for p in (STAGE4_DIR / 'val'   / 'images').glob('*.*'))
assert len(train_names_s4 & val_names_s4) == 0, 'Stage 4 data leak detected!'
print('Data leak check: OK ✓')

model_s4 = YOLO('yolo26s.pt')
results_s4 = model_s4.train(
    data=str(stage4_yaml),
    epochs=100,
    batch=16,
    imgsz=640,
    patience=30,
    project='/kaggle/working/runs',
    name='stage4_infrastructure',
    pretrained=True,
    # Infrastructure detection: moderate augmentation
    mosaic=0.8,
    mixup=0.1,
    degrees=10.0,
    scale=0.4,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    erasing=0.2,
    dropout=0.1,
    weight_decay=0.001,
    label_smoothing=0.05,   # lower — multi-class infra labels are well-defined
    optimizer='SGD',
    lr0=0.01,
    lrf=0.001,
    warmup_epochs=3,
    cos_lr=True,
    momentum=0.937,
    multi_scale=False,
    verbose=True,
    save_period=10,
    plots=True,
)

metrics_s4 = results_s4.results_dict
print(f"Stage 4 mAP50:    {metrics_s4.get('metrics/mAP50(B)', 'N/A')}")
print(f"Stage 4 mAP50-95: {metrics_s4.get('metrics/mAP50-95(B)', 'N/A')}")

---
*Pipeline cells 1–15 complete: datasets extracted, merged, verified, YAMLs written.*
*Proceed to training below.*

In [ ]:
## 4. Save All Models

import shutil
from pathlib import Path
from ultralytics import YOLO

output_dir = WORK_DIR / 'trained_models'
output_dir.mkdir(exist_ok=True)

# Stage 1: pretrained YOLO26n (no training needed)
m = YOLO('yolo26n.pt')
m.save(str(output_dir / 'stage1_person.pt'))
print('Stage 1 saved:', output_dir / 'stage1_person.pt')

# Stage 2
best_s2 = Path('/kaggle/working/runs/stage2_ppe_harness/weights/best.pt')
if best_s2.exists():
    shutil.copy2(best_s2, output_dir / 'stage2_ppe_harness.pt')
    print('Stage 2 (trained) saved')
else:
    YOLO('yolo26s.pt').save(str(output_dir / 'stage2_ppe_harness.pt'))
    print('Stage 2 (pretrained fallback) saved')

# Stage 3
best_s3 = Path('/kaggle/working/runs/stage3_environment/weights/best.pt')
if best_s3.exists():
    shutil.copy2(best_s3, output_dir / 'stage3_environment.pt')
    print('Stage 3 (trained) saved')
else:
    YOLO('yolo26n.pt').save(str(output_dir / 'stage3_environment.pt'))
    print('Stage 3 (pretrained fallback) saved')

# Stage 4
best_s4 = Path('/kaggle/working/runs/stage4_infrastructure/weights/best.pt')
if best_s4.exists():
    shutil.copy2(best_s4, output_dir / 'stage4_infrastructure.pt')
    print('Stage 4 (trained) saved')
else:
    YOLO('yolo26s.pt').save(str(output_dir / 'stage4_infrastructure.pt'))
    print('Stage 4 (pretrained fallback) saved')

print(f'\nAll models saved → {output_dir}')
print('Download via Kaggle Output panel → place in backend/models/')

In [ ]:
## 5. Validate Models

In [ ]:
# ─── Validate All Trained Stages ─────────────────────────────────────────────
# Re-define paths here so this cell can run independently of cell 24
from pathlib import Path
from ultralytics import YOLO

output_dir = Path('/kaggle/working/trained_models')
best_s2    = Path('/kaggle/working/runs/stage2_ppe_harness/weights/best.pt')
best_s3    = Path('/kaggle/working/runs/stage3_environment/weights/best.pt')
best_s4    = Path('/kaggle/working/runs/stage4_infrastructure/weights/best.pt')
stage2_yaml = Path('/kaggle/working/stage2.yaml')
stage3_yaml = Path('/kaggle/working/stage3.yaml')
stage4_yaml = Path('/kaggle/working/stage4.yaml')

def has_test_split(sdir):
    p = Path(sdir) / 'test' / 'images'
    return p.exists() and any(p.glob('*.*'))

# ── Stage 2 ──────────────────────────────────────────────────────────────────
if best_s2.exists():
    s2_model = YOLO(str(output_dir / 'stage2_ppe_harness.pt'))
    split = 'test' if has_test_split('/kaggle/working/stage2') else 'val'
    val_s2 = s2_model.val(data=str(stage2_yaml), split=split)
    print(f'Stage 2 ({split}) mAP50:    {val_s2.box.map50:.4f}')
    print(f'Stage 2 ({split}) mAP50-95: {val_s2.box.map:.4f}')
else:
    print('Stage 2: training not completed, skipping validation.')

# ── Stage 3 ──────────────────────────────────────────────────────────────────
if best_s3.exists():
    s3_model = YOLO(str(output_dir / 'stage3_environment.pt'))
    split = 'test' if has_test_split('/kaggle/working/stage3') else 'val'
    val_s3 = s3_model.val(data=str(stage3_yaml), split=split)
    print(f'Stage 3 ({split}) mAP50:    {val_s3.box.map50:.4f}')
    print(f'Stage 3 ({split}) mAP50-95: {val_s3.box.map:.4f}')
else:
    print('Stage 3: training not completed, skipping validation.')

# ── Stage 4 ──────────────────────────────────────────────────────────────────
if best_s4.exists():
    s4_model = YOLO(str(output_dir / 'stage4_infrastructure.pt'))
    split = 'test' if has_test_split('/kaggle/working/stage4') else 'val'
    val_s4 = s4_model.val(data=str(stage4_yaml), split=split)
    print(f'Stage 4 ({split}) mAP50:    {val_s4.box.map50:.4f}')
    print(f'Stage 4 ({split}) mAP50-95: {val_s4.box.map:.4f}')
else:
    print('Stage 4: training not completed, skipping validation.')